# 1 — Construct and audit a pseudopopulation
Fresh source-level computation. Public default: invented three-landmark/six-pair data. Supplied-data mode uses an explicit protected manifest and requires independent execution approval. No model fitting.

In [ ]:
from neuro.notebook_artifacts import no_fitting, FitAccounting
whole_notebook_no_fitting = no_fitting()
whole_notebook_no_fitting.__enter__()
accounting = FitAccounting().start()
import json, os, sys, time, warnings
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from IPython import get_ipython
get_ipython().run_line_magic('matplotlib', 'inline')
from neuro.notebook_inputs import configuration_from_environment, inspect_inputs, load_inputs, effective_pair_map
from neuro.pseudopopulation import sample_plan, materialize, support
from neuro.notebook_artifacts import ArtifactStore, FitAccounting, target_heads, write_private_json, verify_artifacts, runtime_identity, validate_kernel_identity
from neuro.notebook_support import workspace, usage_summary, source_budget, checked_assemblies, compact_metrics, plot_confusions

configuration = configuration_from_environment()
settings = configuration['execution']
output = workspace()
def private_warning(message, category, filename, lineno, file=None, line=None):
    # Preserve full provenance privately; do not render host paths in shared figures/output.
    warning_path = output / 'warnings-private.jsonl'
    with warning_path.open('a') as handle:
        handle.write(json.dumps({'message': str(message), 'category': category.__name__, 'filename': filename, 'lineno': lineno}) + '\n')
    warning_path.chmod(0o600)
    print(category.__name__ + ': ' + str(message), file=sys.stderr)
warnings.showwarning = private_warning
store = ArtifactStore(output / 'artifacts')
store.save('configuration', configuration)
store.save('kernel_runtime_before_inputs', validate_kernel_identity(configuration))
print('Input mode:', configuration['mode'], '| exploratory computational assay; no biological IID inference')

## Explicit metadata and target boundary
Authenticate only supplied files, regenerate eligibility/assignments, freeze the contributor embargo and complete-support panel before neural interpretation. The public default is invented data. Missing supplied inputs fail; no demo or historical-result fallback.

In [ ]:
inspection = inspect_inputs(configuration)  # Metadata/hash step only for supplied inputs.
store.save('metadata_selection', inspection)  # Protected provenance; never displayed.
print(json.dumps(inspection.audit, sort_keys=True))
assert inspection.audit['contributor_overlap'] == 0
assert inspection.audit['selected_train'] > 0 and inspection.audit['selected_development'] > 0

## A worked target derivation
Previous/current successful adjacent-trial marker digits give provisional start/goal. The adapter checks multimodal triplets, importer agreement, QC and eligibility. Only a pseudonymous worked example is printed; the full joined derivation/exclusions remain protected.

In [ ]:
example = inspection.selected[0]
worked = {'example': 'source_example', 'previous_marker_digit': example['start'], 'current_marker_digit': example['goal'],
          'derived_pair': f"{example['start']}->{example['goal']}", 'status': 'derived/provisional, not independently measured truth'}
print(json.dumps(worked))
assert worked['derived_pair'] == example['condition']
print('Exclusion waterfall:', json.dumps(inspection.audit['exclusion_counts'], sort_keys=True))

## Inventory, eligibility and support are different questions
Compact source mode retains all records, including failed/mismatched trials. The historical outer-train gate, frozen development labels and global predecessor-contributor embargo apply only to consumption. Eye masked placeholders are never observed events. Metadata summaries do not create a pristine holdout.

In [ ]:
from collections import Counter
landmarks = configuration['landmarks']
pair_counts = Counter(s['condition'] for s in inspection.eligible)
selected_counts = Counter(s['condition'] for s in inspection.selected)
print('Before/after source counts:', len(inspection.eligible), len(inspection.selected))
print('Input inventory:', {k: inspection.audit[k] for k in ('exported_trials', 'exported_units', 'unity_timing_rows', 'native_eye_missing_events') if k in inspection.audit})
print('Metadata all-record outcomes (historically exposed, not a holdout):', inspection.audit.get('all_record_outcomes', {}))
for title, counts in [('Eligible outer-train ordered pairs', pair_counts), ('Selected ordered pairs', selected_counts)]:
    matrix = np.array([[counts[f'{a}->{b}'] for b in landmarks] for a in landmarks])
    print(title, '(rows=start, columns=goal; diagonal/unsupported cells remain zero):'); print(matrix)
    fig, ax = plt.subplots(figsize=(5, 4)); image = ax.imshow(matrix); fig.colorbar(image, ax=ax, label='Real source trials')
    ax.set(xticks=range(len(landmarks)), xticklabels=landmarks, yticks=range(len(landmarks)), yticklabels=landmarks, xlabel='Goal', ylabel='Start', title=title)
    display(fig); plt.close(fig)
# Every original block/pair appears, including zero surviving training support.
eligible = inspection.eligible
contributors = {sid for s in eligible if s['split_id'] == 'development' for sid in (s['source_id'], s['previous_id'])}
surviving = [s for s in eligible if s['split_id'] == 'development' or not {s['source_id'], s['previous_id']} & contributors]
pools = Counter((s['block_id'], s['condition'], s['split_id']) for s in surviving)
blocks = sorted({s['block_id'] for s in eligible})
pairs = [f'{a}->{b}' for a in landmarks for b in landmarks if a != b]
block_support = np.array([[pools[block, pair, 'train'] for pair in pairs] for block in blocks])
print('All original block/pair training support (zeros retained):', block_support.tolist())
print('Predecessor edges:', inspection.audit.get('directed_edges', {}))
print('Development contributor union / embargo removed:', len(contributors), inspection.audit['embargo_removed_train'])
if inspection.place195_selection is not None:
    joined = inspection.place195_selection.trials[example['source_id']]
    print('Worked marker join:', {k: joined[k] for k in ('ripple_codes', 'unity_codes', 'eyelink_codes', 'eye_observed', 'outcome', 'qc_flags') if k in joined})
    print('Native-precision cue seconds:', joined['ripple_cue_duration_s'])

## Fresh neural measurements, observation and exposure
The compact adapter consumes only the selected outer-train panel, retains repeated native spikes, and uses explicit absolute half-open boundaries without the historical binning epsilon. Full-precision results need not equal old nine-significant-digit CSV results. The older six-file adapter retains its own conservation checks; neither route loads fitted results. Whole-file hashes traverse all supplied bytes, not physical non-reading.

In [ ]:
data, neural_audit = load_inputs(inspection)  # Supplied neural path remains explicitly review-gated.
store.save('source_data', data)
store.save('neural_audit', neural_audit)
print(json.dumps(neural_audit, sort_keys=True))
assert np.isfinite(data.values[data.observed]).all()
assert np.isnan(data.values[~data.observed]).all()
assert not data.dependency_audit()['overlapping_contributor_ids']
print('Source × unit × bin shape:', data.values.shape, '| units:', data.value_unit, '| clock units:', data.time_unit)

## One raw spike train → ten cue bins
The input stores unbinned native timestamps, not cached counts. The exact boundary policy is visible below; zeros are observed only within the source's recorded block, while off-block values remain masked/NaN.

In [ ]:
if configuration['mode'] == 'compact':
    from neuro.cue_source import authenticate, exact_bins
    root, dataset_manifest = authenticate(inspection.place195_selection.manifest)
    first_source = inspection.selected[0]
    ui = next(i for i, u in enumerate(inspection.units) if u['session_id'] == first_source['block_id'])
    unit = inspection.units[ui]
    trial = inspection.place195_selection.trials[first_source['source_id']]
    with np.load(root / 'spikes.npz', allow_pickle=False) as archive:
        native_ms = archive[unit['key'] + '__timestamps']
    counts, tail, full = exact_bins(native_ms, trial['ripple_start_s'], trial['ripple_end_s'], data.bin_edges)
    np.testing.assert_array_equal(counts, data.values[0, ui])
    print('Cue edges (seconds from onset):', data.bin_edges.tolist())
    print('Fresh counts:', counts.tolist(), '| omitted cue tail:', tail, '| all cue spikes:', full)
print('Observed zero count bins:', int(np.sum(data.values[data.observed] == 0)))
print('Off-block unavailable bins:', int(np.isnan(data.values).sum()))

In [ ]:
from neuro.notebook_support import plot_input_audit
fig = plot_input_audit(inspection, data, neural_audit)
display(fig); plt.close(fig)
print('Selected block×condition×split support: min/max', min(r['source_count'] for r in support(data)), max(r['source_count'] for r in support(data)))

## An intact assembled row and exact reconstruction
One source block supplies all of that block’s units and bins. Unit order is preserved. Plans bind observed input bytes separately from effective conditions. All ID-bearing lineage is saved privately.

In [ ]:
effective = effective_pair_map(data, permutation_seed=None)
plan = sample_plan(data, split_id='train', rows_per_condition=settings['train_rows_per_condition'], seed=0, effective_conditions=effective)
assembly = materialize(data, plan)
reconstructed = materialize(data, assembly.plan)
np.testing.assert_array_equal(assembly.values, reconstructed.values)
np.testing.assert_array_equal(assembly.targets, reconstructed.targets)
store.save('construction_assembly', assembly)
print('Constructed rows:', len(plan.rows), '| intact blocks per row:', len(data.block_ids))
print('First row aliases:', {f'block_{i+1:02d}': 'contributing_source' for i in range(len(data.block_ids))})
fig, ax = plt.subplots(figsize=(8, 3))
illustration = assembly.values[0] if configuration['mode'] == 'invented' else np.nanmean(data.values, axis=0)
image = ax.imshow(illustration, aspect='auto'); fig.colorbar(image, ax=ax, label='Count per bin' if configuration['mode'] == 'invented' else 'Mean count per bin')
ax.set(xlabel='Native cue bin', ylabel='Stable unit index', title='Invented intact row' if configuration['mode'] == 'invented' else 'Aggregate unit/bin means across selected sources')
display(fig); plt.close(fig)

## Eligible / used / unused / repeated source accounting
The constructed-row count is not an independent biological sample size.

In [ ]:
usage = usage_summary(data, assembly)
print(json.dumps(usage, sort_keys=True))
fig, ax = plt.subplots(figsize=(6, 3)); ax.bar(list(usage['reuse_histogram']), list(usage['reuse_histogram'].values()))
ax.set(xlabel='Uses of a real source', ylabel='Real sources', title='Source reuse distribution')
display(fig); plt.close(fig)

## Interpretation limits
These are source-safe conditional pseudopopulations, not simultaneous trials. Matching uses known labels; this is not an unknown-trial prediction pipeline. Bins, synthetic rows and computational seeds are dependent. Place-selected units, provisional marker meaning, inherited preprocessing and historical exposure remain limitations. Contributor separation does not remove long-range serial/session confounds. No biological confidence intervals, p-values, mechanism, dynamics or prospective-path claim follows from these figures. The permutation diagnostic is alignment-breaking, not a biological exchangeability test.

In [ ]:
fit_counts = accounting.stop()
store.save('kernel_runtime_final', runtime_identity())
expected_counts = {'pca': 0, 'logistic': 0, 'lda': 0, 'cebra': 0, 'optimizer_updates': 0}
manifest_hash = store.finish(fit_counts=fit_counts, expected_counts=expected_counts)
replay = verify_artifacts(output / 'artifacts', expected_manifest_hash=manifest_hash)
write_private_json(output / 'completion.json', {'manifest_sha256': manifest_hash, 'fit_counts': fit_counts, 'replay': replay})
print(json.dumps(replay, sort_keys=True))
assert replay['status'] == 'fit-blocked-loaded-model-replay-passed'
whole_notebook_no_fitting.__exit__(None, None, None)